# 금융감독원 API로 금융상품 데이터 수집
https://finlife.fss.or.kr/finlife/main/contents.do?menuNo=700029

# 1. API 호출하고 DataFrame 변환
* 은행 권역코드 : 020000 = 은행, 030300 = 저축은행, 030200 = 상호금융

## API_KEY

In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")
# print(API_KEY)

## 적금

In [2]:
url = "http://finlife.fss.or.kr/finlifeapi/savingProductsSearch.json"
groups = ["020000", "030300", "030200"]
all_data = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_saving = pd.DataFrame(all_data)

print(df_saving.shape)

(333, 15)


* 컬럼 이름 한글로 바꾸기

In [3]:
df_saving = df_saving.rename(columns={
    "dcls_month": "공시월",
    "fin_co_no": "금융회사코드",
    "fin_prdt_cd": "금융상품코드",
    "kor_co_nm": "금융회사명",
    "fin_prdt_nm": "금융상품명",
    "join_way": "가입방법",
    "mtrt_int": "만기후이자율",
    "spcl_cnd": "우대조건",
    "join_deny": "가입제한",
    "join_member": "가입대상",
    "etc_note": "기타유의사항",
    "max_limit": "최고한도",
    "dcls_strt_day": "공시시작일",
    "dcls_end_day": "공시종료일",
    "fin_co_subm_day": "금융회사제출일"
})

In [4]:
df_saving

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,만기후이자율,우대조건,가입제한,가입대상,기타유의사항,최고한도,공시시작일,공시종료일,금융회사제출일
0,202603,0010001,WR0001F,우리은행,우리SUPER주거래적금,"영업점,인터넷,스마트폰,전화(텔레뱅킹)",만기 후\n- 1개월이내 : 만기시점약정이율×50%\n- 1개월초과 6개월이내: 만...,1. 거래실적 인정기간 동안 우리은행 입출식 계좌에서 아래 각 항목별 실적이 있는 ...,1,실명의 개인,1. 가입기간 : 1년/2년/3년\n2. 가입금액 : 월 50만원 이내,NaN,20260320,NaN,202603201027
1,202603,0010001,WR0001L,우리은행,WON적금,"스마트폰,전화(텔레뱅킹)",만기 후\n- 1개월이내 : 만기시점약정이율×50%\n- 1개월초과 6개월이내: 만...,"1. 아래 각 항(가, 나)의 조건을 충족하는 경우 합산 최대 연 0.2%p 우대\...",1,실명의 개인,1. 가입기간 : 1년\n2. 가입금액 : 월 50만원 이내,NaN,20260320,NaN,202603201027
2,202603,0010002,00266451,한국스탠다드차타드은행,퍼스트가계적금,"영업점,인터넷,스마트폰",만기 후 1개월: 약정이율의 50%\n만기 후 1개월 초과 1년 이내: 약정이율의 ...,없음,1,개인(개인사업자 포함),해당없음,10000000.0,20260320,99991231,202603201012
3,202603,0010016,10521001001166004,아이엠뱅크,iM함께적금,"영업점,인터넷,스마트폰",만기 후 1개월 미만 경과: 약정이자율 x 50%\n만기 후 3개월 미만 경과: 약...,*최고우대금리:연0.85%p\n-전월 총수신 평잔 30만원 이상 또는 첫만남플러스통...,1,실명의 개인 및 개인사업자,계좌당 가입 최저한도 : 10만원,NaN,20260320,NaN,202603200952
4,202603,0010017,01020400490002,부산은행,펫 적금,"영업점,스마트폰",- 만기후 1년이내:가입기간별 일반정기적금 기본이율 x 50%\n- 만기후 1년초과...,"*우대이율 6개월제 최대 0.55%, 12개월제 최대 0.90%",1,실명의 개인고객(1인 1계좌),1. 가입한도: 월 1만원 이상 50만원 이하 원단위\n2. 정기적립식,500000.0,20260320,NaN,202603201000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,202603,0013350,1130315008,웰컴저축은행,WELCOME 아이사랑 정기적금,"영업점,인터넷,스마트폰","*만기 후 1개월 이하 : 이 예금의 가입 당시 약정금리나 만기시점의 동일 상품, ...",① 계약기간의 2/3회 이상 웰컴 입출금통장을 통한 자동이체로 납입 : 1.0%p ...,1,제한없음,1인당 가입한도 월 10만원 (1인 1계좌),100000.0,20260220,NaN,202602201200
329,202603,0013350,1130315009,웰컴저축은행,웰뱅하자 정기적금,"영업점,인터넷,스마트폰","*만기 후 1개월 이하 : 이 예금의 가입 당시 약정금리나 만기시점의 동일 상품, ...",1) 웰컴 입출금통장에서 CMS/지로 자동납부 월 2건이상 실적이 계약기간의 2/3...,1,제한없음,1인당 가입한도 월 20만원 (1인 1계좌),200000.0,20260220,NaN,202602201200
330,202603,0013350,1130315011,웰컴저축은행,웰컴 페이적금,"영업점,인터넷,스마트폰","*만기 후 1개월 이하 : 이 예금의 가입 당시 약정금리나 만기시점의 동일 상품, ...",1) 계약기간의 2/3회 이상 웰컴 입출금통장을 통한 자동이체로 납입 : 2.0%p...,1,제한없음,"1인당 가입한도 월 30만원 (1인 1계좌, WELCOME 체크플러스2 정기적금 보...",300000.0,20260220,NaN,202602201200
331,202603,0013351,310001,OK저축은행,OK정기적금,영업점,-만기 후 1개월 이하: 약정이율 혹은 만기시점 해당상품 이율 중 낮은 이율 \n-...,없음,1,제한없음,1. 가입금액: 1만원 이상\n2. 영업점전용상품,NaN,20260220,NaN,202602201200


In [9]:
df_saving["금융상품명"].str.contains("연금저축", na=False).any()

np.False_

In [7]:
url = "http://finlife.fss.or.kr/finlife/fdrmDpstApi.json"
groups = ["020000", "030300", "030200"]
all_data10 = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data2.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_list = pd.DataFrame(all_data10)

print(df_list.shape)

(0, 0)


## 예금

In [8]:
url = "http://finlife.fss.or.kr/finlifeapi/depositProductsSearch.json"
groups = ["020000", "030300", "030200"]
all_data2 = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data2.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_deposit = pd.DataFrame(all_data2)

print(df_deposit.shape)

(424, 15)


In [10]:
df_deposit = df_deposit.rename(columns={
    "dcls_month": "공시월",
    "fin_co_no": "금융회사코드",
    "fin_prdt_cd": "금융상품코드",
    "kor_co_nm": "금융회사명",
    "fin_prdt_nm": "금융상품명",
    "join_way": "가입방법",
    "mtrt_int": "만기후이자율",
    "spcl_cnd": "우대조건",
    "join_deny": "가입제한",
    "join_member": "가입대상",
    "etc_note": "기타유의사항",
    "max_limit": "최고한도",
    "dcls_strt_day": "공시시작일",
    "dcls_end_day": "공시종료일",
    "fin_co_subm_day": "금융회사제출일"
})

In [11]:
df_deposit

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,만기후이자율,우대조건,가입제한,가입대상,기타유의사항,최고한도,공시시작일,공시종료일,금융회사제출일
0,202603,0010001,WR0001B,우리은행,WON플러스예금,"인터넷,스마트폰,전화(텔레뱅킹)",만기 후\n- 1개월이내 : 만기시점약정이율×50%\n- 1개월초과 6개월이내: 만...,해당사항 없음,1,실명의 개인,"- 가입기간: 1~36개월\n- 최소가입금액: 1만원 이상\n- 만기일을 일,월 단...",NaN,20260320,NaN,202603201027
1,202603,0010002,00320342,한국스탠다드차타드은행,e-그린세이브예금,"인터넷,스마트폰",만기 후 1개월: 약정이율의 50%\n만기 후 1개월 초과 1년 이내: 약정이율의 ...,1.SC제일은행 최초 거래 신규고객에 대하여 우대 이율을 제공함 (보너스이율0.2%...,1,개인(개인사업자 포함),"디지털채널 전용상품 (인터넷, 모바일뱅킹)",1.000000e+09,20260330,99991231,202603300948
2,202603,0010016,10511008001166004,아이엠뱅크,iM함께예금,"영업점,인터넷,스마트폰",만기 후 1개월 미만 경과 : 약정이자율 x 50%\n만기 후 3개월 미만 경과 :...,* 최고우대금리: 연0.45%p\n- 전월 총수신 평잔 30만원 이상 또는 상품 가...,1,실명의 개인 및 개인사업자,계좌당 가입 최저한도 : 100만원,NaN,20260320,NaN,202603200950
3,202603,0010017,01030500510002,부산은행,LIVE정기예금,"영업점,인터넷","- 만기후1년내: 가입기간별 일반정기예금이율 x 50%,\n- 만기후1년초과:가입기...",*우대이율\n가. 3~5개월 특판우대이율 : 0.70%\n나. 6~11개월 특판 우...,1,제한없음,1. 가입금액 :\n 1천만원 이상\n2. 가입기간 : \n1개월 이상 60개월...,NaN,20260320,NaN,202603200959
4,202603,0010017,01030500560002,부산은행,더(The) 특판 정기예금,"인터넷,스마트폰","- 만기후1년내: 가입기간별 일반정기예금이율 x 50%,\n- 만기후1년초과:가입기...",* 우대이율 (최대 0.90%p)\n가. 모바일뱅킹 금융정보 및 혜택알림 동의 우대...,1,실명의 개인,"1. 가입금액 : 1백만원 이상 제한없음 (원단위)\n2. 가입기간 : 1개월, 3...",NaN,20260320,NaN,202603200959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,202603,0013351,240067,OK저축은행,OK e-정기예금,"인터넷,스마트폰","만기 후 1개월 이하 약정이율 혹은 만기시점 해당상품 이율 중 낮은 이율, 만기 후...",없음,1,제한없음,-,1.000000e+10,20260303,NaN,202603030841
420,202603,0013351,240070,OK저축은행,OK안심정기예금(변동금리),영업점,"만기 후 1개월 이하 약정이율 혹은 만기시점 해당상품 이율 중 낮은 이율, 만기 후...",당행 보통예금(활동계좌) 보유 시 +0.1%p,1,제한없음,3년제 정기예금으로 가입 후 매 1년 마다 해당시점 금리로 자동연장되는 변동금리 상...,1.000000e+10,20260331,NaN,202603310833
421,202603,0013351,240071,OK저축은행,OK e-안심정기예금(변동금리),"인터넷,스마트폰","만기 후 1개월 이하 약정이율 혹은 만기시점 해당상품 이율 중 낮은 이율, 만기 후...",없음,1,제한없음,3년제 정기예금으로 가입 후 매 1년 마다 해당시점 금리로 자동연장되는 변동금리 상...,1.000000e+10,20260331,NaN,202603310833
422,202603,0013351,240097,OK저축은행,OK e-안심앱플러스정기예금(변동금리),스마트폰,"만기 후 1개월 이하: 약정이율 혹은 만기시점 해당상품 이율 중 낮은 이율, 만기 ...",없음,1,개인,*OK저축은행 모바일뱅킹앱 전용상품\n*3년제 정기예금으로 가입 후 매 1년 마다 ...,1.000000e+10,20260220,NaN,202602201200


## 주택담보대출

In [12]:
url = "http://finlife.fss.or.kr/finlifeapi/mortgageLoanProductsSearch.json"
groups = ["020000", "030300", "030200"]
all_data3 = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data3.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_mortgageLoan = pd.DataFrame(all_data3)

print(df_mortgageLoan.shape)

(91, 13)


In [15]:
df_mortgageLoan = df_mortgageLoan.rename(columns={
    "dcls_month": "공시월",
    "fin_co_no": "금융회사코드",
    "fin_prdt_cd": "금융상품코드",
    "kor_co_nm": "금융회사명",
    "fin_prdt_nm": "금융상품명",
    "join_way": "가입방법",
    "loan_inci_expn": "대출부대비용",
    "erly_rpay_fee": "중도상환수수료",
    "dly_rate": "연체이자율",
    "loan_lmt": "대출한도",
    "dcls_strt_day": "공시시작일",
    "dcls_end_day": "공시종료일",
    "fin_co_subm_day": "금융회사제출일"
})

In [16]:
df_mortgageLoan.head()

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,대출부대비용,중도상환수수료,연체이자율,대출한도,공시시작일,공시종료일,금융회사제출일
0,202603,0010001,1054,우리은행,우리아파트론,"영업점,모집인",- 인지세 : 해당세액의 50%(대출금액 5천만원 이하시 없음)\n- 국민주택채권 ...,- 고정금리 : 중도상환대출금×0.71%×잔존기간÷대출기간\n- 변동금리 : 중도상...,- 적용금리+ 3%\n(가계대출 최고연체이자율 : 12%),LTV 70%,20260320,20270319,202603191704
1,202603,0010001,1055,우리은행,우리부동산론,"영업점,모집인",- 인지세 : 해당세액의 50%(대출금액 5천만원 이하시 없음)\n- 국민주택채권 ...,- 고정금리 : 중도상환대출금×0.71%×잔존기간÷대출기간\n- 변동금리 : 중도상...,- 적용금리+ 3%\n(가계대출 최고연체이자율 : 12%),LTV 70%,20260320,20270319,202603191704
2,202603,0010002,SC002111SC002015,한국스탠다드차타드은행,주택담보대출,"영업점,스마트폰",인지세 : 해당세액의 50% (대출금액 5천만원 이하 시 없음)\n국민주택채권 매입...,조기상환원금 × 0.4% × [(3년-대출경과일수) / 3년]\n매년 대출잔액의 1...,대출금리 + 연 3%\n(최고 연체이자율 : 연 15%),LTV 최대 80%,20260319,NaN,202603191932
3,202603,0010016,20283770000001001,아이엠뱅크,HYBRID 모기지론(생활),"영업점,모집인",1.인지세:해당세액의 50%\n2.국민주택채권매입비용 : 대출금액×115%×1%×채...,중도상환대출금액×0.51%×(3년-대출경과일수)÷3년(일수),○ 대출이자율 + 3.0%\n○ 최고 연체이자율 : 15.0%,담보인정비율(LTV) 70%,20260320,20260430,202603180853
4,202603,0010016,20283770001162002,아이엠뱅크,IM주택담보대출_5년고정금리(생활),"영업점,모집인",1.인지세:해당세액의 50%\n2.국민주택채권매입비용 : 대출금액×115%×1%×채...,중도상환대출금액×0.51%×(3년-대출경과일수)÷3년(일수),○ 대출이자율 + 3.0%\n○ 최고 연체이자율 : 15.0%,담보인정비율(LTV) 70%,20260320,20260430,202603180853


## 전세자금대출

In [17]:
url = "http://finlife.fss.or.kr/finlifeapi/rentHouseLoanProductsSearch.json"
groups = ["020000", "030300", "030200"]
all_data4 = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data4.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_rentHouseLoan = pd.DataFrame(all_data4)

print(df_rentHouseLoan.shape)

(48, 13)


In [19]:
df_rentHouseLoan = df_rentHouseLoan.rename(columns={
    "dcls_month": "공시월",
    "fin_co_no": "금융회사코드",
    "fin_prdt_cd": "금융상품코드",
    "kor_co_nm": "금융회사명",
    "fin_prdt_nm": "금융상품명",
    "join_way": "가입방법",
    "loan_inci_expn": "대출부대비용",
    "erly_rpay_fee": "중도상환수수료",
    "dly_rate": "연체이자율",
    "loan_lmt": "대출한도",
    "dcls_strt_day": "공시시작일",
    "dcls_end_day": "공시종료일",
    "fin_co_subm_day": "금융회사제출일"
})

In [21]:
df_rentHouseLoan.head()

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,대출부대비용,중도상환수수료,연체이자율,대출한도,공시시작일,공시종료일,금융회사제출일
0,202603,0010001,10561,우리은행,우리전세론(주택금융보증),"영업점,모집인",- 인지세 : 해당세액의 50%(대출금액 5천만원 이하시 없음)\n- 보증료,- 고정금리 : 중도상환대출금×0.76%×잔존기간÷대출기간\n- 변동금리 : 중도상...,- 적용금리+ 3%\n(가계대출 최고연체이자율 : 12%),최대4.44억원,20260320,20270319,202603191705
1,202603,0010001,10562,우리은행,우리전세론(서울보증),"영업점,모집인","- 인지세 : 해당세액의 50%(대출금액 5천만원 이하시 없음)\n- 질권설정, 채...",- 고정금리 : 중도상환대출금×0.76%×잔존기간÷대출기간\n- 변동금리 : 중도상...,- 적용금리+ 3%\n(가계대출 최고연체이자율 : 12%),최대5억원,20260320,20270319,202603191705
2,202603,0010002,SC002013SC002003,한국스탠다드차타드은행,전세담보대출,"영업점,스마트폰",인지세 : 해당세액의 50% (대출금액 5천만원 이하 시 없음),중도상환금액× 0.2%× (대출잔여일수 ÷ 대출기간 총일수),대출금리 + 연 3%\n(최고 연체이자율 : 연 15%),최대 5억원,20260319,NaN,202603191931
3,202603,0010016,20460801000001001,아이엠뱅크,DGB 전세자금대출,"영업점,모집인",1. 인지세 : 해당세액의 50%\n2. 주택금융보증료 : 연 0.12% ~ 0.40%,"중도상환대출금액×(고정금리:0.57, 변동금리:0.47)%×(대출잔여일수 ÷ 3년)",○ 대출이자율 + 3.0%\n○ 최고 연체이자율: 15.0%,440백만원,20260320,20260430,202603180853
4,202603,0010016,20460801000001002,아이엠뱅크,무방문전세자금대출(주택금융공사),"영업점,모집인","1. 인지세 : 해당세액의 50%\n2. 질권설정통지수수료 :30,000원","중도상환대출금액×(고정금리:0.57, 변동금리:0.47)%×(대출잔여일수 ÷ 3년)",○ 대출이자율 + 3.0%\n○ 최고 연체이자율: 15.0%,220백만원,20260320,20260430,202603180853


## 신용대출

In [22]:
url = "http://finlife.fss.or.kr/finlifeapi/creditLoanProductsSearch.json"
groups = ["020000", "030300", "030200"]
all_data5 = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data5.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_creditLoan = pd.DataFrame(all_data5)

print(df_creditLoan.shape)

(107, 12)


In [25]:
df_creditLoan = df_creditLoan.rename(columns={
    "dcls_month": "공시월",
    "fin_co_no": "금융회사코드",
    "fin_prdt_cd": "금융상품코드",
    "crdt_prdt_type": "신용상품유형코드",
    "kor_co_nm": "금융회사명",
    "fin_prdt_nm": "금융상품명",
    "join_way": "가입방법",
    "cb_name": "신용평가사명",
    "crdt_prdt_type_nm": "신용상품유형명",
    "dcls_strt_day": "공시시작일",
    "dcls_end_day": "공시종료일",
    "fin_co_subm_day": "금융회사제출일"
})

In [26]:
df_creditLoan.head()

,공시월,금융회사코드,금융상품코드,신용상품유형코드,금융회사명,금융상품명,가입방법,신용평가사명,신용상품유형명,공시시작일,공시종료일,금융회사제출일
0,202603,0010001,CR0001A,1,우리은행,협약금리 外 신용대출상품,"영업점,인터넷,스마트폰",KCB,일반신용대출,20260319,99991231,202603181721
1,202603,0010001,CR0001C,2,우리은행,협약금리 外 신용대출상품,"영업점,인터넷,스마트폰",KCB,마이너스한도대출,20260319,99991231,202603181721
2,202603,0010002,SC001217,1,한국스탠다드차타드은행,개인신용대출,"영업점,스마트폰",KCB,일반신용대출,20260319,NaN,202603191345
3,202603,0010002,SC001217_1,2,한국스탠다드차타드은행,개인신용대출,"영업점,스마트폰",KCB,마이너스한도대출,20260319,NaN,202603191345
4,202603,0010006,WR0002F,3,한국씨티은행,장기카드대출,"인터넷,스마트폰,전화(텔레뱅킹),기타",KCB,장기카드대출(카드론),20260321,NaN,202603121506


## 연금저축

In [8]:
url = "http://finlife.fss.or.kr/finlifeapi/annuitySavingProductsSearch.json"
groups = ["020000", "030300", "030200"]
all_data6 = []

for g in groups:
    page = 1
    while True:
        try:
            res = requests.get(url, params={
                "auth": API_KEY,
                "topFinGrpNo": g,
                "pageNo": page
            }).json()
        except:
            break

        result = res.get("result")
        if not result:
            break

        data = result.get("baseList")
        if not data:
            break

        # 리스트에 바로 누적
        all_data5.extend(data)

        page += 1

# 마지막에 한 번만 DataFrame 생성
df_annuitySaving = pd.DataFrame(all_data6)

print(df_annuitySaving.shape)

(0, 0)


# 2. 전처리

In [50]:
df_deposit[df_deposit["가입제한"] == 0]

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,만기후이자율,우대조건,가입제한,가입대상,기타유의사항,최고한도,공시시작일,공시종료일,금융회사제출일


In [54]:
df_deposit[df_deposit["최고한도"].notnull()].head(1)

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,만기후이자율,우대조건,가입제한,가입대상,기타유의사항,최고한도,공시시작일,공시종료일,금융회사제출일
1,202603,0010002,00320342,한국스탠다드차타드은행,e-그린세이브예금,"인터넷,스마트폰",만기 후 1개월: 약정이율의 50%\n만기 후 1개월 초과 1년 이내: 약정이율의 ...,1.SC제일은행 최초 거래 신규고객에 대하여 우대 이율을 제공함 (보너스이율0.2%...,1,개인(개인사업자 포함),"디지털채널 전용상품 (인터넷, 모바일뱅킹)",1.000000e+09,20260330,99991231,202603300948


* df_saving, df_deposit : 최고한도 Null => 한도 없음 / 가입제한 1 => 가입제한 컬럼에 1만 존재하므로 컬럼 삭제 / 금융회사제출일 => 불필요하므로 삭제 
* df_mortgageLoan : 금융회사제출일 => 불필요하므로 삭제 
* df_rentHouseLoan : 금융회사제출일 => 불필요하므로 삭제 
* df_creditLoan : 금융회사제출일 => 불필요하므로 삭제 

* 공시시작일/공시종료일 : 실제 상품의 시작일/종료일이 아님. 따라서, 종료일이 지나면 제거. 

In [56]:
# 1. df_saving, df_deposit 처리
# 최고한도 null → 한도 없음으로 대체
df_saving["최고한도"] = df_saving["최고한도"].fillna("한도 없음")
df_deposit["최고한도"] = df_deposit["최고한도"].fillna("한도 없음")

# 가입제한 컬럼 삭제 (값이 1만 존재)
df_saving = df_saving.drop(columns=["가입제한"])
df_deposit = df_deposit.drop(columns=["가입제한"])

# 금융회사제출일 삭제
df_saving = df_saving.drop(columns=["금융회사제출일"])
df_deposit = df_deposit.drop(columns=["금융회사제출일"])

# 2. 나머지 df들 → 금융회사제출일 삭제
dfs = [df_mortgageLoan, df_rentHouseLoan, df_creditLoan]

df_mortgageLoan, df_rentHouseLoan, df_creditLoan = [df.drop(columns=["금융회사제출일"]) for df in dfs]

In [57]:
dfs = [df_saving, df_deposit, df_mortgageLoan, df_rentHouseLoan, df_creditLoan]

for df in dfs:
    display(df.head(1))

,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,만기후이자율,우대조건,가입대상,기타유의사항,최고한도,공시시작일,공시종료일
0,202603,0010001,WR0001F,우리은행,우리SUPER주거래적금,"영업점,인터넷,스마트폰,전화(텔레뱅킹)",만기 후\n- 1개월이내 : 만기시점약정이율×50%\n- 1개월초과 6개월이내: 만...,1. 거래실적 인정기간 동안 우리은행 입출식 계좌에서 아래 각 항목별 실적이 있는 ...,실명의 개인,1. 가입기간 : 1년/2년/3년\n2. 가입금액 : 월 50만원 이내,한도 없음,20260320,NaN


,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,만기후이자율,우대조건,가입대상,기타유의사항,최고한도,공시시작일,공시종료일
0,202603,0010001,WR0001B,우리은행,WON플러스예금,"인터넷,스마트폰,전화(텔레뱅킹)",만기 후\n- 1개월이내 : 만기시점약정이율×50%\n- 1개월초과 6개월이내: 만...,해당사항 없음,실명의 개인,"- 가입기간: 1~36개월\n- 최소가입금액: 1만원 이상\n- 만기일을 일,월 단...",한도 없음,20260320,NaN


,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,대출부대비용,중도상환수수료,연체이자율,대출한도,공시시작일,공시종료일
0,202603,0010001,1054,우리은행,우리아파트론,"영업점,모집인",- 인지세 : 해당세액의 50%(대출금액 5천만원 이하시 없음)\n- 국민주택채권 ...,- 고정금리 : 중도상환대출금×0.71%×잔존기간÷대출기간\n- 변동금리 : 중도상...,- 적용금리+ 3%\n(가계대출 최고연체이자율 : 12%),LTV 70%,20260320,20270319


,공시월,금융회사코드,금융상품코드,금융회사명,금융상품명,가입방법,대출부대비용,중도상환수수료,연체이자율,대출한도,공시시작일,공시종료일
0,202603,0010001,10561,우리은행,우리전세론(주택금융보증),"영업점,모집인",- 인지세 : 해당세액의 50%(대출금액 5천만원 이하시 없음)\n- 보증료,- 고정금리 : 중도상환대출금×0.76%×잔존기간÷대출기간\n- 변동금리 : 중도상...,- 적용금리+ 3%\n(가계대출 최고연체이자율 : 12%),최대4.44억원,20260320,20270319


,공시월,금융회사코드,금융상품코드,신용상품유형코드,금융회사명,금융상품명,가입방법,신용평가사명,신용상품유형명,공시시작일,공시종료일
0,202603,0010001,CR0001A,1,우리은행,협약금리 外 신용대출상품,"영업점,인터넷,스마트폰",KCB,일반신용대출,20260319,99991231


In [61]:
dfs = [df_saving, df_deposit, df_mortgageLoan, df_rentHouseLoan, df_creditLoan]

for df in dfs:
    df["공시시작일"] = pd.to_datetime(df["공시시작일"], errors="coerce")
    df["공시종료일"] = pd.to_datetime(df["공시종료일"], errors="coerce")

In [62]:
df_saving['공시시작일'].dtype

dtype('<M8[us]')

* 공시시작일/종료일은 상품 판매기간이 아니라 **금리·조건이 적용된 “공시 버전의 유효기간**
* 금리 변경 시 기존 공시는 종료되고 같은 상품코드로 새로운 공시가 생성됨 (최신 공시 = 현재 상태)

따라서 데이터 처리는 공시종료일이 지난 데이터는 제거하고 같은 금융상품코드 기준으로 가장 최신 공시(공시시작일 기준) 1개만 남긴다

In [64]:
def clean(df):
    df["공시시작일"] = pd.to_datetime(df["공시시작일"], errors="coerce")
    df["공시종료일"] = pd.to_datetime(df["공시종료일"], errors="coerce")

    today = pd.to_datetime("today")

    return (df[(df["공시종료일"].isnull()) | (df["공시종료일"] >= today)]
            .sort_values("공시시작일")
            .drop_duplicates("금융상품코드", keep="last"))

df_saving, df_deposit, df_mortgageLoan, df_rentHouseLoan, df_creditLoan = [clean(df) for df in [df_saving, df_deposit, df_mortgageLoan, df_rentHouseLoan, df_creditLoan]]

# 3. CSV 파일로 각각 저장

In [67]:
df_saving.to_csv("./data/df_saving.csv", index=False, encoding="utf-8-sig")
df_deposit.to_csv("./data/df_deposit.csv", index=False, encoding="utf-8-sig")
df_mortgageLoan.to_csv("./data/df_mortgageLoan.csv", index=False, encoding="utf-8-sig")
df_rentHouseLoan.to_csv("./data/df_rentHouseLoan.csv", index=False, encoding="utf-8-sig")
df_creditLoan.to_csv("./data/df_creditLoan.csv", index=False, encoding="utf-8-sig")